## Studying effects of Actvations, Gradients and Batch Norm on the MLP model

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
words = open("../data.txt", "r").read().splitlines()

In [ ]:
stoi = {chr(i+97): i+1 for i in range(26)}
stoi['.'] = 0

itos = {value : key for key, value in stoi.items()}

In [ ]:
vocabsize = 27

In [ ]:
block_size = 3

def build_dataset(data):
    X, Y = [], []

    for w in data:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)

            context = context[1:] + [ix]

    return torch.tensor(X), torch.tensor(Y)

import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtrain, ytrain = build_dataset(words[:n1])
Xval, yval = build_dataset(words[n1:n2])
Xtest, ytest = build_dataset(words[n2:])

In [ ]:
nembedd = 10
nhidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocabsize, nembedd), generator=g)
W1 = torch.randn(((nembedd*block_size), nhidden), generator=g) * [1, 0.2][1] # to reduce chances of dead neurons at beginning by reducing saturation of h at tails
b1 = torch.randn(nhidden, generator=g) * 0.01   # to allow a bit of variation and entropy at the start when no optimized vector is known
W2 = torch.randn((nhidden, vocabsize), generator=g) * 0.01 # scaling the weights for better initialisation of NN so that training is more optimized
b2 = torch.randn(vocabsize, generator=g) * 0    # to improve the initialisation of the nn


W1 = torch.randn(((nembedd*block_size), nhidden), generator=g) * (5/3) * ((nembedd*block_size) ** (-0.5))   # Kaiming init
"""
    The W2 matrix was scaled to 0.01 instead of 0
    The b2 matrix was set to 0 for the first iteration as we dont want to add a random noise to the network at the start

    This improves the softmax layer and improves the results
"""

parameters = [C, W1, b1, W2, b2]

print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

Changed Results: (log losses)

```
1. Original
    train: 2.21453
    val: 2.16819
2. Fix softmax confidently wrong(W2 and b2 init)
    train: 2.07
    val: 2.13
3. Fix tanh layer too saturated at init
    train: 2.0615
    val: 2.1129
```

for deeper networks these results would be more contrasting as they start to stack up layer by layer growing exponentially

In [ ]:
lr = 0.1
epochs = 100000
batchsize = 32
lossi = []

for i in range(epochs):
    ix = torch.randint(0, Xtrain.shape[0], (batchsize, ), generator=g)
    Xb, yb = Xtrain[ix], ytrain[ix]

    emb = C[Xb]

    h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, yb)

    for p in parameters:
        p.grad = None
    loss.backward()

    if i == epochs/2: lr = 0.01
    for p in parameters:
        p.data -= lr * p.grad

    if (i % 10000) == 0: print(f"epoch {i}/{epochs}: {loss}")
    lossi.append(loss.log10().item())

    #break

print(f"Loss: {loss}")


In [ ]:
# problems with the current solution at the hidden layer (W1, b1)
plt.hist(h.view(-1).tolist(), 50);

"""
    the tanh fxn grad (1 - h**2) at the tails ie around -1 and 1, gets 0 ie p.grad = 0 at some h = -1 or 1
    therefore for the current distribution of h(after first iteration), which is heavily concentrated at the tails 
    the parameter updation for a few neurons might be dead ie stuck at the same value 
    if all the h's going to a neuron are either -1 or 1
    or the updation of parametric values will be very low if its tending towards -1 and 1
    ie it will make convergence of loss function impossible or very slow
"""

In [ ]:
# therefore the saturation of h on tails need to be removed or that is the preactivations(emb @ W1 + b1) need to be set closer to 0

# 1. Scale the weights W1, b1 closer to zero
# 2. This will make the h = emb @ W1 + b1 more saturated around zero and reducing the concentration on the tails(-1, 1)

In [ ]:
plt.figure(figsize=(20, 10))
plt.imshow(h.abs() > 0.99, cmap = 'gray', interpolation='nearest')

""" 
    If there is any column which is entirely white(ie all the h entering that neurons are either tending towords -1 or 1) then that neuron will not update and will be dead
    Therefore, our aim is to reduce the white cells 
    {White cells represent those h that are > 0.99 in this example}
"""

In [ ]:
plt.plot(lossi)

In [ ]:
@torch.no_grad()    # decorator to stop gradient tracking

def splitloss(split):
    x, y = {
        'train' : (Xtrain, ytrain),
        'val' : (Xval, yval),
        'test' : (Xtest, ytest)
    }[split]

    emb = C[x]
    h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)

    return loss

print(splitloss('train'))
print(splitloss('val'))

In [ ]:
# sampling

for i in range(10):
    word = ""
    context = [0]*block_size
    ix = 1

    while(ix):
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
        logits = h @ W2 + b2
        prob = torch.softmax(logits, dim = 1)

        ix = torch.multinomial(prob, 1, replacement=True, generator=g).item()

        word += itos[ix]
        context = context[1:] + [ix]

    print(word)

#### Kaiming init: 
Intialize the weight matrix --> <b>W = torch.randn(n, m) * (gain / (sq. root(n)))</b>

ie, Gaussian distribution weights scaled by gain multiplied by the inverse of square root of n. For<br>
1. tanh: gain = 5/3
2. relu: gain = 2**0.5
<br>

this preseves the std dev of the training distribution: ``` X.stddev = approx(X @ W_kaiming).stddev ```

this is an important result as we always want to preserve the real world distribution of data and generalise it using the model

In [ ]:
# Whatever we multiply a gaussian distribution with becomes its square root
# torch.randn(1000).std() is close to 1
# but (torch.randn(1000) * n).std() is close to n
# Therefore, the kaiming initialisation is used as a standard practice

<b>Batch Normalisation</b>

In [ ]:
nembedd = 10
nhidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocabsize, nembedd), generator=g)
W1 = torch.randn(((nembedd*block_size), nhidden), generator=g) * (5/3) * ((nembedd*block_size) ** (-0.5))   # Kaiming init
b1 = torch.randn(nhidden, generator=g) * 0.01
W2 = torch.randn((nhidden, vocabsize), generator=g) * 0.01
b2 = torch.randn(vocabsize, generator=g) * 0

bngain = torch.ones((1, nhidden))   # Gamma based on BN paper
bnbias = torch.zeros((1, nhidden))   # B based on BN paper

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
# Batch Normalisation

lr = 0.1
epochs = 100000
batchsize = 32
lossi = []

for i in range(epochs):
    ix = torch.randint(0, Xtrain.shape[0], (batchsize, ), generator=g)
    Xb, yb = Xtrain[ix], ytrain[ix]

    emb = C[Xb]

    hreact = emb.view(emb.shape[0], -1) @ W1 + b1
    # accelerates the convergence of NN at the start of training
    hreact_norm = (hreact - hreact.mean(0, keepdim = True))/hreact.std(0, keepdim = True)      # standardisation
    hreact_norm = bngain * hreact_norm + bnbias # scale and shift based on BN paper

    h = torch.tanh(hreact_norm)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, yb)

    for p in parameters:
        p.grad = None
    loss.backward()

    if i == epochs/2: lr = 0.01
    for p in parameters:
        p.data -= lr * p.grad

    if (i % 10000) == 0: print(f"epoch {i}/{epochs}: {loss}")
    lossi.append(loss.log10().item())

    #break

print(f"Loss: {loss}")


In [ ]:
# while batch normalisation is good for training it is not good for calibrating test as it couples each example with other samples 
# form the same batch therefore what we call a data leak might happen 
# therefore for normalising during val and test we use the same mean and stddev we used while training

with torch.no_grad():
    emb = C[Xtrain]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1

    bnmean = hpreact.mean(0, keepdim = True)
    bnstd = hpreact.std(0, keepdim = True)

In [ ]:
@torch.no_grad()    # decorator to stop gradient tracking

def splitloss(split):
    x, y = {
        'train' : (Xtrain, ytrain),
        'val' : (Xval, yval),
        'test' : (Xtest, ytest)
    }[split]

    emb = C[x]
    hreact = emb.view(emb.shape[0], -1) @ W1 + b1
    hreact_norm = (hreact - bnmean)/bnstd
    hreact_norm = bngain * hreact_norm + bnbias
    h = torch.tanh(hreact_norm)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)

    return loss

print(splitloss('train'))
print(splitloss('val'))

<b>Deeper NN and Torchified code</b>

In [ ]:
class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None: self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

C = torch.randn((vocabsize, n_embd),            generator=g)
layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocabsize, bias=False), BatchNorm1d(vocabsize),
]
# layers = [
#   Linear(n_embd * block_size, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, vocab_size),
# ]

with torch.no_grad():
  # last layer: make less confident
  layers[-1].gamma *= 0.1
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True
        

In [ ]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtrain.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtrain[ix], ytrain[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function
  
  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('activation distribution')

# saturation should be low and stabilised
# tanh is a squashing function and therfore some gain needs to be multiplied to it during normalisation
# therefore too small of a gain would squash to zero and
# too large of a gain would saturate to tails therfore an appropriate gain is used for non linear layers

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('gradient distribution')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution');

In [ ]:
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:
    plt.plot([ud[j][i] for j in range(len(ud))])
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot
plt.legend(legends);

In [ ]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtrain, ytrain),
    'val': (Xval, yval),
    'test': (Xtest, ytest),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  x = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, y)
  print(split, loss.item())

# put layers into eval mode
for layer in layers:
  layer.training = False
split_loss('train')
split_loss('val')

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass the neural net
      emb = C[torch.tensor([context])] # (1,block_size,n_embd)
      x = emb.view(emb.shape[0], -1) # concatenate the vectors
      for layer in layers:
        x = layer(x)
      logits = x
      probs = F.softmax(logits, dim=1)
      # sample from the distribution
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      # shift the context window and track the samples
      context = context[1:] + [ix]
      out.append(ix)
      # if we sample the special '.' token, break
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out)) # decode and print the generated word